In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-4 ; m=1.0 ; g=9.81 ;
tf=1.5; N=int(tf/dt);

t=np.linspace(0,tf,N)


X_val = 3.0      # X  (m)
P_val = 8.0      # P  (kg m/s)

# Condiciones iniciales (x0, p0): apartados (a)-(d) y un caso extra (lanzamiento hacia arriba, x hacia abajo)
casos = [
    {"label": "(a)  $x_0=0,\\ p_0=0$",       "x0": 0.0,   "p0": 0.0,    "color": "tab:blue"},
    {"label": "(b)  $x_0=X,\\ p_0=0$",       "x0": X_val, "p0": 0.0,    "color": "tab:green"},
    {"label": "(c)  $x_0=X,\\ p_0=P$",       "x0": X_val, "p0": P_val,  "color": "tab:red"},
    {"label": "(d)  $x_0=0,\\ p_0=P$",       "x0": 0.0,   "p0": P_val,  "color": "tab:purple"},
    {"label": "extra  $x_0=X,\\ p_0=-P$",    "x0": X_val, "p0": -P_val, "color": "tab:orange"},
]
x0s = np.array([c["x0"] for c in casos])
p0s = np.array([c["p0"] for c in casos])

## ECUACIONES DE HAMILTON (H = p·ẋ − L)

In [ ]:
#Lagrangiano, momento y Hamiltoniano   (H = p ẋ − L)      x medida hacia abajo
x_s, xd_s, p_s = sp.symbols('x xdot p', real=True)
m_, g_ = sp.symbols('m g', positive=True)

T_sym = sp.Rational(1,2)*m_*xd_s**2
U_sym = -m_*g_*x_s                                  # x hacia abajo: la gravedad empuja hacia x creciente
L_sym = T_sym - U_sym
E_sym = T_sym + U_sym

p_expr = sp.diff(L_sym, xd_s)                       # p = ∂L/∂ẋ   (derivada parcial)
H_vel  = sp.simplify(p_expr*xd_s - L_sym)           # H = p ẋ − L   (en función de x, ẋ)

print('L =', L_sym)
print('p =', p_expr)
print('H(x, ẋ) = p ẋ − L =', H_vel)
print('H − E =', sp.simplify(H_vel - E_sym), '   ->  H = E' if sp.simplify(H_vel - E_sym) == 0 else '   ->  H ≠ E')

# Hamiltoniano en variables canónicas: ẋ = ẋ(p)
sol   = sp.solve(sp.Eq(p_s, p_expr), xd_s, dict=True)[0]
H_can = sp.simplify(H_vel.subs(sol))
print('H(x, p) =', H_can)

# Ecuaciones de Hamilton
xdot_eq = sp.diff(H_can, p_s)                       #  ẋ =  ∂H/∂p
pdot_eq = -sp.diff(H_can, x_s)                      #  ṗ = −∂H/∂x
print('ẋ =', xdot_eq, ' ,  ṗ =', pdot_eq)

f_num = sp.lambdify((x_s, p_s, m_, g_), [xdot_eq, pdot_eq], 'numpy')
H_num = sp.lambdify((x_s, p_s, m_, g_), H_can, 'numpy')

# Órbitas en el espacio de fases:  H(x, p) = E  ->  x(p)
E_s = sp.symbols('E', real=True)
x_orbit_sym = sp.solve(sp.Eq(H_can, E_s), x_s)[0]
print('Órbita: x(p) =', x_orbit_sym)
x_orbit_num = sp.lambdify((p_s, E_s, m_, g_), x_orbit_sym, 'numpy')

## FUNCION PARA EL MOVIMIENTO

In [ ]:
def Motion(x0, p0, pars, t):
    """Integra las ecuaciones de Hamilton para varios casos a la vez (Euler predictor + Heun corrector)."""
    N, nc = len(t), len(x0)
    x, p = np.zeros((nc, N)), np.zeros((nc, N))
    x[:, 0], p[:, 0] = x0, p0

    for i in range(N - 1):
        dt = t[i+1] - t[i]

        # --- PREDICTOR (Euler) ---
        xd_i, pd_i = f_num(x[:, i], p[:, i], *pars)
        x_pred = x[:, i] + xd_i*dt
        p_pred = p[:, i] + pd_i*dt

        # --- Derivadas en el punto predicho ---
        xd_n, pd_n = f_num(x_pred, p_pred, *pars)

        # --- CORRECTOR (Heun) ---
        x[:, i+1] = x[:, i] + 0.5*(xd_i + xd_n)*dt
        p[:, i+1] = p[:, i] + 0.5*(pd_i + pd_n)*dt

    return x, p


#Simulación
pars = (m, g)
x, p = Motion(x0s, p0s, pars, t)

## SOLUCIÓN ANALÍTICA Y ENERGÍA

In [ ]:
#Solución analítica:   ẋ = p/m ,  ṗ = m g     ->     p(t) = p0 + m g t ,   x(t) = x0 + (p0/m) t + ½ g t²
p_an = p0s[:, None] + m*g*t[None, :]
x_an = x0s[:, None] + (p0s[:, None]/m)*t[None, :] + 0.5*g*t[None, :]**2

#Energía (H = E se conserva):   E = p²/2m − m g x
E = H_num(x, p, *pars)

print(f'{"caso":<8}{"E0 (J)":>10}{"max|E−E0|":>14}{"max|x−x_an|":>16}{"max|p−p_an|":>16}')
for c, nm in enumerate(["(a)", "(b)", "(c)", "(d)", "extra"]):
    print(f'{nm:<8}{E[c,0]:>10.3f}{np.abs(E[c]-E[c,0]).max():>14.2e}{np.abs(x[c]-x_an[c]).max():>16.2e}{np.abs(p[c]-p_an[c]).max():>16.2e}')

## ESPACIO DE FASES

In [ ]:
#Espacio de fases: órbitas desde t=0 hasta t=tf
t_mark = 0.25*np.arange(0, int(tf/0.25) + 1)            # marcas de tiempo (s) sobre cada órbita
i_mark = np.clip(np.round(t_mark/dt).astype(int), 0, N - 1)

xmin, xmax = x.min() - 2, x.max() + 2
pmin, pmax = p.min() - 3, p.max() + 3

fig, ax = plt.subplots(figsize=(10, 7))

# Campo de flujo (ẋ = p/m, ṗ = m g)
xg, pg = np.linspace(xmin, xmax, 300), np.linspace(pmin, pmax, 300)
Xg, Pg = np.meshgrid(xg, pg)
ax.streamplot(xg, pg, Pg/m, np.full_like(Xg, m*g), color='gainsboro', density=1.2, linewidth=0.9, arrowsize=1.2)

p_full = np.linspace(pmin, pmax, 600)
for c, caso in enumerate(casos):
    col = caso["color"]
    # órbita completa (tenue) y tramo recorrido entre t=0 y t=tf (grueso)
    ax.plot(x_orbit_num(p_full, E[c, 0], m, g), p_full, '--', color=col, lw=1, alpha=0.4)
    ax.plot(x[c], p[c], color=col, lw=2.5, label=caso["label"])
    ax.scatter(x[c, i_mark], p[c, i_mark], s=18, color='white', edgecolors=col, zorder=4)       # marcas de tiempo
    ax.scatter(x[c, 0], p[c, 0], s=80, color=col, edgecolors='black', zorder=5)                 # condición inicial
    ax.annotate('', xy=(x[c, -1], p[c, -1]), xytext=(x[c, -400], p[c, -400]),
                arrowprops=dict(arrowstyle='-|>', color=col, lw=2, mutation_scale=18), zorder=5)  # sentido de recorrido

ax.scatter([], [], s=18, color='white', edgecolors='k', label=f'marcas cada {t_mark[1]:.2f} s')
ax.axhline(0, color='k', ls=':', lw=1); ax.axvline(0, color='k', ls=':', lw=1)
ax.set(xlim=(xmin, xmax), ylim=(pmin, pmax), xlabel='Posición generalizada $x$ (m, hacia abajo)',
       ylabel='Momento conjugado $p$ (kg m/s)', title=f'Espacio de fases: masa bajo gravedad  (órbitas de $t=0$ a $t={tf}$ s)')
ax.grid(alpha=0.4, ls='--'); ax.legend(loc='upper left', framealpha=0.9)
plt.tight_layout(); plt.show()


#Un panel por apartado
fig, axs_ = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for c, a in enumerate(axs_.ravel()):
    col = casos[c]["color"]
    a.streamplot(xg, pg, Pg/m, np.full_like(Xg, m*g), color='gainsboro', density=0.9, linewidth=0.8, arrowsize=1)
    a.plot(x_orbit_num(p_full, E[c, 0], m, g), p_full, '--', color=col, lw=1, alpha=0.4)
    a.plot(x[c], p[c], color=col, lw=2.5)
    a.scatter(x[c, i_mark], p[c, i_mark], s=18, color='white', edgecolors=col, zorder=4)
    a.scatter(x[c, 0], p[c, 0], s=80, color=col, edgecolors='black', zorder=5)
    a.annotate('', xy=(x[c, -1], p[c, -1]), xytext=(x[c, -400], p[c, -400]),
               arrowprops=dict(arrowstyle='-|>', color=col, lw=2, mutation_scale=18), zorder=5)
    a.axhline(0, color='k', ls=':', lw=1); a.axvline(0, color='k', ls=':', lw=1)
    a.set(xlim=(xmin, xmax), ylim=(pmin, pmax)); a.set_title(casos[c]["label"]); a.grid(alpha=0.4, ls='--')
for a in axs_[1]: a.set_xlabel('$x$ (m)')
for a in axs_[:, 0]: a.set_ylabel('$p$ (kg m/s)')
plt.tight_layout(); plt.show()

## x(t) Y p(t)

In [ ]:
#x(t) y p(t): numérica frente a analítica
s = max(1, N // 5000)                                    # submuestreo

fig, ax = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for c, caso in enumerate(casos):
    ax[0,0].plot(t[::s], x[c, ::s], color=caso["color"], lw=2, label=caso["label"])
    ax[0,0].plot(t[::s], x_an[c, ::s], 'k:', lw=1)
    ax[0,1].plot(t[::s], p[c, ::s], color=caso["color"], lw=2)
    ax[0,1].plot(t[::s], p_an[c, ::s], 'k:', lw=1)
    ax[1,0].plot(t[::s], (x[c] - x_an[c])[::s], color=caso["color"])
    ax[1,1].plot(t[::s], (p[c] - p_an[c])[::s], color=caso["color"])
ax[0,0].invert_yaxis()                                   # x medida hacia abajo
ax[0,0].set(ylabel='x (m)  (hacia abajo)', title='Posición  (línea punteada: analítica)'); ax[0,0].legend(fontsize=7)
ax[0,1].set(ylabel='p (kg m/s)', title='Momento')
ax[1,0].set(xlabel='t (s)', ylabel='x_num − x_an'); ax[1,1].set(xlabel='t (s)', ylabel='p_num − p_an')
for a in ax.ravel(): a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ANIMACIÓN

In [ ]:
#Animación: espacio de fases (izquierda) + movimiento real en la vertical (derecha)
plt.rcParams['animation.embed_limit'] = 100        # MB

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 20       # fotogramas por segundo
speed   = 0.5      # 1 = tiempo real, 0.5 = cámara lenta

# Submuestreo: un fotograma cada 'step' pasos de simulación
step = max(1, round(speed / (fps * dt)))
idx  = np.arange(0, min(N, int(T_anim / dt)), step)
ts   = t[idx]
xa, pa = x[:, idx], p[:, idx]
nc = len(casos)

fig, (A1, A2) = plt.subplots(1, 2, figsize=(12, 5.5), gridspec_kw={'width_ratios': [1.6, 1]})

# --- Espacio de fases ---
A1.streamplot(xg, pg, Pg/m, np.full_like(Xg, m*g), color='gainsboro', density=1.0, linewidth=0.8, arrowsize=1)
for c, caso in enumerate(casos):
    A1.plot(x_orbit_num(p_full, E[c, 0], m, g), p_full, '--', color=caso["color"], lw=1, alpha=0.35)
A1.axhline(0, color='k', ls=':', lw=1); A1.axvline(0, color='k', ls=':', lw=1)
A1.set(xlim=(xmin, xmax), ylim=(pmin, pmax), xlabel='$x$ (m, hacia abajo)', ylabel='$p$ (kg m/s)', title='Espacio de fases')
A1.grid(alpha=0.4, ls='--')

# --- Movimiento real: x hacia abajo, cada caso en su "carril" ---
A2.set(xlim=(-0.8, nc - 0.2), ylim=(xmax, xmin), title='Movimiento vertical ($x$ hacia abajo)', ylabel='$x$ (m)')
A2.set_xticks(range(nc)); A2.set_xticklabels(["(a)", "(b)", "(c)", "(d)", "extra"])
A2.axhline(0, color='k', ls=':', lw=1); A2.grid(alpha=0.3)

trails, dots, bodies = [], [], []
for c, caso in enumerate(casos):
    trails.append(A1.plot([], [], '-', color=caso["color"], lw=2.5, label=caso["label"])[0])
    dots.append(A1.plot([], [], 'o', color=caso["color"], mec='black', ms=9)[0])
    bodies.append(A2.plot([], [], 'o', color=caso["color"], mec='black', ms=12)[0])
A1.legend(loc='upper left', fontsize=8, framealpha=0.9)
time_txt = A1.text(0.98, 0.03, '', transform=A1.transAxes, ha='right')

def update(k):
    for c in range(nc):
        trails[c].set_data(xa[c, :k+1], pa[c, :k+1])
        dots[c].set_data([xa[c, k]], [pa[c, k]])
        bodies[c].set_data([c], [xa[c, k]])
    time_txt.set_text(f't = {ts[k]:.2f} s')
    return (*trails, *dots, *bodies, time_txt)

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())